In [2]:
# Install faker library directly from Jupyter
import sys
!{sys.executable} -m pip install faker prophet shap

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 12.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   --------------- ------------------------ 4.7/12.1 MB 24.6 MB/s eta 0:00:01
   ------------------------- -------------- 7.6/12.1 MB 24.2 MB/s eta 0:00:01
   ---------------------------------------- 12.1/12.1 MB 21.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 17.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   --------------------------------------- 549.1/549.1 kB 13.9 MB/s eta 0:00:00

   ---------- ----------------------------- 2/8 [importlib_resources]
   ---------- ----------------------------- 2/8 [importlib_resources]
   --------------- ------------------------ 3/8 [faker]
   --------------- ------------------------ 3/8 [faker]

In [3]:
# ============================================================
# SUPPLY CHAIN INTELLIGENCE SYSTEM
# Notebook 1: Dataset Generator
# ============================================================

import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility - so data is same every time you run
np.random.seed(42)
random.seed(42)
fake = Faker('en_IN')  # Indian locale for realistic names
Faker.seed(42)

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

✅ All libraries imported successfully!
Pandas version: 2.2.3
NumPy version: 2.1.3


In [4]:
# ============================================================
# TABLE 1: PRODUCTS
# 50 SKUs across 5 categories - realistic Indian FMCG/D2C brand
# ============================================================

categories = {
    'Staples': ['Basmati Rice 5kg', 'Toor Dal 1kg', 'Chana Dal 1kg', 'Wheat Flour 10kg', 
                'Mustard Oil 1L', 'Sunflower Oil 1L', 'Sugar 1kg', 'Salt 1kg',
                'Poha 500g', 'Sooji 500g'],
    'Personal Care': ['Cold Press Coconut Oil 500ml', 'Neem Face Wash 100ml', 
                      'Herbal Shampoo 200ml', 'Aloe Vera Gel 150g', 'Turmeric Soap 100g',
                      'Rose Water 200ml', 'Hair Oil 200ml', 'Body Lotion 300ml',
                      'Lip Balm 10g', 'Face Scrub 100g'],
    'Snacks': ['Roasted Chana 200g', 'Mathri 400g', 'Namkeen Mix 300g', 
               'Rice Crackers 150g', 'Dry Fruits Mix 250g', 'Protein Bar 60g',
               'Makhana 100g', 'Baked Chips 150g', 'Trail Mix 200g', 'Granola 300g'],
    'Beverages': ['Green Tea 100 bags', 'Masala Chai Premix 500g', 'Turmeric Latte 200g',
                  'Instant Coffee 100g', 'Protein Shake Mix 500g', 'Electrolyte Powder 200g',
                  'Herbal Juice 1L', 'Coconut Water 200ml', 'Rose Sherbet 750ml', 
                  'Aam Panna Mix 400g'],
    'Household': ['Neem Floor Cleaner 1L', 'Dish Wash Liquid 500ml', 'Laundry Liquid 1L',
                  'Toilet Cleaner 500ml', 'Glass Cleaner 500ml', 'Handwash 250ml',
                  'Room Freshener 200ml', 'Mosquito Repellent 100ml',
                  'Surface Disinfectant 500ml', 'Fabric Softener 500ml']
}

# Pricing by category (cost, selling price ranges in INR)
price_config = {
    'Staples':       {'cost': (80, 320),   'price': (120, 480),  'shelf_life': (180, 540)},
    'Personal Care': {'cost': (120, 450),  'price': (200, 750),  'shelf_life': (365, 730)},
    'Snacks':        {'cost': (60, 280),   'price': (100, 450),  'shelf_life': (60, 180)},
    'Beverages':     {'cost': (90, 380),   'price': (150, 600),  'shelf_life': (90, 365)},
    'Household':     {'cost': (70, 260),   'price': (120, 420),  'shelf_life': (365, 730)}
}

products = []
sku_counter = 1

for category, product_list in categories.items():
    cfg = price_config[category]
    for product_name in product_list:
        unit_cost = round(random.uniform(*cfg['cost']), 2)
        selling_price = round(unit_cost * random.uniform(1.4, 1.9), 2)  # 40-90% margin
        shelf_life = random.randint(*cfg['shelf_life'])
        
        # Reorder point based on category velocity
        reorder_point = random.randint(200, 1000)
        moq = reorder_point * random.randint(2, 5)  # MOQ always higher than ROP
        
        products.append({
            'sku_id': f'SKU{str(sku_counter).zfill(3)}',
            'product_name': product_name,
            'category': category,
            'unit_cost_inr': unit_cost,
            'selling_price_inr': selling_price,
            'gross_margin_pct': round((selling_price - unit_cost) / selling_price * 100, 2),
            'shelf_life_days': shelf_life,
            'reorder_point_units': reorder_point,
            'minimum_order_qty': moq,
            'is_active': 1
        })
        sku_counter += 1

df_products = pd.DataFrame(products)

# Save to raw folder
df_products.to_csv("D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Raw/products.csv", index=False)

print(f"✅ Products table created successfully!")
print(f"Total SKUs: {len(df_products)}")
print(f"\nCategory breakdown:")
print(df_products['category'].value_counts())
print(f"\nSample data:")
print(df_products.head(3).to_string())

✅ Products table created successfully!
Total SKUs: 50

Category breakdown:
category
Staples          10
Personal Care    10
Snacks           10
Beverages        10
Household        10
Name: count, dtype: int64

Sample data:
   sku_id      product_name category  unit_cost_inr  selling_price_inr  gross_margin_pct  shelf_life_days  reorder_point_units  minimum_order_qty  is_active
0  SKU001  Basmati Rice 5kg  Staples         233.46             329.76             29.20              320                  450               1350          1
1  SKU002      Toor Dal 1kg  Staples         113.49             164.70             31.09              459                  289               1445          1
2  SKU003     Chana Dal 1kg  Staples          87.63             126.79             30.89              299                  717               1434          1


In [5]:
# ============================================================
# TABLE 2: SUPPLIERS
# 15 suppliers across India - realistic names, cities, categories
# 3 suppliers intentionally made risky (we will "discover" this later)
# ============================================================

supplier_data = [
    # (name, city, state, category, lead_time, payment_terms, reliability_tier)
    # reliability_tier: 'high', 'medium', 'low' — we use this to simulate realistic data
    
    ('Agro Fresh Pvt Ltd',          'Amritsar',   'Punjab',           'Staples',       7,  30, 'high'),
    ('GrainMart Industries',        'Indore',     'Madhya Pradesh',   'Staples',       10, 45, 'medium'),
    ('PureFarm Commodities',        'Karnal',     'Haryana',          'Staples',       8,  30, 'high'),
    ('NatureCare Products',         'Coimbatore', 'Tamil Nadu',       'Personal Care', 12, 45, 'high'),
    ('HerbalRoot Mfg Co',           'Pune',       'Maharashtra',      'Personal Care', 15, 60, 'medium'),
    ('GreenLeaf Essentials',        'Ahmedabad',  'Gujarat',          'Personal Care', 14, 45, 'low'),   # RISKY
    ('CrunchBite Foods Pvt Ltd',    'Rajkot',     'Gujarat',          'Snacks',        9,  30, 'high'),
    ('SnackFactory India',          'Ludhiana',   'Punjab',           'Snacks',        11, 30, 'low'),   # RISKY
    ('MunchWell Industries',        'Nashik',     'Maharashtra',      'Snacks',        10, 45, 'medium'),
    ('BrewMaster Beverages',        'Bengaluru',  'Karnataka',        'Beverages',     13, 60, 'high'),
    ('TeaLeaf Exports Ltd',         'Siliguri',   'West Bengal',      'Beverages',     18, 45, 'medium'),
    ('DrinkWell Co',                'Chennai',    'Tamil Nadu',       'Beverages',     16, 60, 'low'),   # RISKY
    ('CleanHome Chemicals',         'Surat',      'Gujarat',          'Household',     8,  30, 'high'),
    ('PureClean Mfg Ltd',           'Hyderabad',  'Telangana',        'Household',     10, 45, 'medium'),
    ('HomeGuard Industries',        'Kanpur',     'Uttar Pradesh',    'Household',     12, 30, 'high'),
]

suppliers = []
for i, (name, city, state, category, lead_time, payment_terms, tier) in enumerate(supplier_data, 1):
    
    # Contract started between 1 to 4 years ago
    contract_start = datetime(2024, 1, 1) - timedelta(days=random.randint(365, 1460))
    
    # Annual capacity based on tier
    capacity_map = {'high': (50000, 100000), 'medium': (25000, 50000), 'low': (10000, 30000)}
    annual_capacity = random.randint(*capacity_map[tier])
    
    # Credit limit based on payment terms
    credit_limit = payment_terms * random.randint(500, 2000) * 100
    
    suppliers.append({
        'supplier_id':          f'SUP{str(i).zfill(3)}',
        'supplier_name':        name,
        'city':                 city,
        'state':                state,
        'category_supplied':    category,
        'avg_lead_time_days':   lead_time,
        'payment_terms_days':   payment_terms,
        'annual_capacity_units': annual_capacity,
        'credit_limit_inr':     credit_limit,
        'contract_start_date':  contract_start.strftime('%Y-%m-%d'),
        'reliability_tier':     tier,
        'is_active':            1
    })

df_suppliers = pd.DataFrame(suppliers)

# Save to raw folder
df_suppliers.to_csv("D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Raw/suppliers.csv", index=False)

print("✅ Suppliers table created successfully!")
print(f"Total Suppliers: {len(df_suppliers)}")
print(f"\nReliability breakdown:")
print(df_suppliers['reliability_tier'].value_counts())
print(f"\nSuppliers by State:")
print(df_suppliers['state'].value_counts())
print(f"\nRISKY Suppliers flagged:")
risky = df_suppliers[df_suppliers['reliability_tier'] == 'low']
print(risky[['supplier_id', 'supplier_name', 'category_supplied', 'city']].to_string())

✅ Suppliers table created successfully!
Total Suppliers: 15

Reliability breakdown:
reliability_tier
high      7
medium    5
low       3
Name: count, dtype: int64

Suppliers by State:
state
Gujarat           3
Punjab            2
Maharashtra       2
Tamil Nadu        2
Madhya Pradesh    1
Haryana           1
Karnataka         1
West Bengal       1
Telangana         1
Uttar Pradesh     1
Name: count, dtype: int64

RISKY Suppliers flagged:
   supplier_id         supplier_name category_supplied       city
5       SUP006  GreenLeaf Essentials     Personal Care  Ahmedabad
7       SUP008    SnackFactory India            Snacks   Ludhiana
11      SUP012          DrinkWell Co         Beverages    Chennai


In [6]:
# ============================================================
# TABLE 3: PURCHASE ORDERS
# 2000+ rows | Jan 2023 - Dec 2024 (2 full years of data)
# Risky suppliers (SUP006, SUP008, SUP012) have:
#   - Higher delay rates
#   - Higher defect rates
#   - More partial deliveries
# ============================================================

from collections import defaultdict

# Reload products and suppliers
df_products  = pd.read_csv(r'D:\Projects\End-to-end projects\8. Supply Chain Intelligence\Data\Raw\products.csv')
df_suppliers = pd.read_csv(r'D:\Projects\End-to-end projects\8. Supply Chain Intelligence\Data\Raw\suppliers.csv')

# Create mapping dictionaries
supplier_tier_map  = df_suppliers.set_index('supplier_id')['reliability_tier'].to_dict()
supplier_lead_map  = df_suppliers.set_index('supplier_id')['avg_lead_time_days'].to_dict()
supplier_cat_map   = df_suppliers.set_index('supplier_id')['category_supplied'].to_dict()
product_cost_map   = df_products.set_index('sku_id')['unit_cost_inr'].to_dict()
product_cat_map    = df_products.set_index('sku_id')['category'].to_dict()

# Group SKUs by category
category_skus = defaultdict(list)
for sku, cat in product_cat_map.items():
    category_skus[cat].append(sku)

print("✅ Data loaded successfully!")
print(f"Products: {len(df_products)} | Suppliers: {len(df_suppliers)}")

✅ Data loaded successfully!
Products: 50 | Suppliers: 15


In [7]:
# ============================================================
# Behaviour config per reliability tier
# ============================================================

tier_config = {
    'high': {
        'delay_prob':        0.08,
        'delay_days_range':  (1, 5),
        'defect_rate_range': (0.00, 0.02),
        'fill_rate_range':   (0.95, 1.00),
        'cancel_prob':       0.01
    },
    'medium': {
        'delay_prob':        0.20,
        'delay_days_range':  (2, 10),
        'defect_rate_range': (0.02, 0.06),
        'fill_rate_range':   (0.88, 0.97),
        'cancel_prob':       0.03
    },
    'low': {
        'delay_prob':        0.45,
        'delay_days_range':  (5, 21),
        'defect_rate_range': (0.05, 0.15),
        'fill_rate_range':   (0.70, 0.90),
        'cancel_prob':       0.07
    }
}

print("✅ Tier config set!")

✅ Tier config set!


In [8]:
# ============================================================
# Generate Purchase Orders
# ============================================================

purchase_orders = []
po_counter = 1

start_date = datetime(2023, 1, 1)
end_date   = datetime(2024, 12, 31)

for supplier_row in df_suppliers.itertuples():
    sup_id    = supplier_row.supplier_id
    sup_tier  = supplier_row.reliability_tier
    sup_cat   = supplier_row.category_supplied
    lead_time = supplier_row.avg_lead_time_days
    cfg       = tier_config[sup_tier]

    # Number of POs per supplier — high tier gets more orders
    po_count_map = {'high': 160, 'medium': 120, 'low': 80}
    num_pos      = po_count_map[sup_tier]

    available_skus = category_skus.get(sup_cat, [])
    if not available_skus:
        continue

    for _ in range(num_pos):
        days_range  = (end_date - start_date).days
        order_date  = start_date + timedelta(days=random.randint(0, days_range - 30))
        promised_date = order_date + timedelta(days=lead_time)

        rand = random.random()

        if rand < cfg['cancel_prob']:
            po_status            = 'Cancelled'
            actual_delivery_date = None
            delay_days           = None
            ordered_qty          = random.randint(500, 3000)
            received_qty         = 0
            defect_qty           = 0

        else:
            is_delayed = random.random() < cfg['delay_prob']

            if is_delayed:
                delay_days           = random.randint(*cfg['delay_days_range'])
                actual_delivery_date = promised_date + timedelta(days=delay_days)
                po_status            = 'Delivered'
            else:
                delay_days           = 0
                actual_delivery_date = promised_date - timedelta(days=random.randint(0, 2))
                po_status            = 'Delivered'

            ordered_qty  = random.randint(500, 5000)
            fill_rate    = random.uniform(*cfg['fill_rate_range'])
            received_qty = int(ordered_qty * fill_rate)

            if received_qty < ordered_qty:
                po_status = 'Partial' if random.random() < 0.4 else 'Delivered'

            defect_rate = random.uniform(*cfg['defect_rate_range'])
            defect_qty  = int(received_qty * defect_rate)

        sku_id    = random.choice(available_skus)
        unit_cost = product_cost_map[sku_id]
        po_value  = round(ordered_qty * unit_cost, 2)

        purchase_orders.append({
            'po_id':                  f'PO{str(po_counter).zfill(5)}',
            'supplier_id':            sup_id,
            'sku_id':                 sku_id,
            'order_date':             order_date.strftime('%Y-%m-%d'),
            'promised_delivery_date': promised_date.strftime('%Y-%m-%d'),
            'actual_delivery_date':   actual_delivery_date.strftime('%Y-%m-%d') if actual_delivery_date else None,
            'ordered_qty':            ordered_qty,
            'received_qty':           received_qty,
            'defect_qty':             defect_qty,
            'unit_cost_inr':          unit_cost,
            'po_value_inr':           po_value,
            'delay_days':             delay_days,
            'po_status':              po_status,
        })
        po_counter += 1

df_po = pd.DataFrame(purchase_orders)

df_po.to_csv("D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Raw/purchase_orders.csv", index=False)

print("✅ Purchase Orders table created successfully!")
print(f"Total POs generated: {len(df_po)}")
print(f"\nPO Status breakdown:")
print(df_po['po_status'].value_counts())
print(f"\nTotal procurement value: ₹{df_po['po_value_inr'].sum():,.0f}")
print(f"\nDelay rate by supplier tier:")
delivered = df_po[df_po['po_status'].isin(['Delivered','Partial'])].copy()
delivered = delivered.merge(df_suppliers[['supplier_id','reliability_tier']], on='supplier_id')
print(delivered.groupby('reliability_tier')['delay_days'].apply(
    lambda x: f"{(x>0).mean()*100:.1f}% delayed").to_string())

✅ Purchase Orders table created successfully!
Total POs generated: 1960

PO Status breakdown:
po_status
Delivered    1163
Partial       756
Cancelled      41
Name: count, dtype: int64

Total procurement value: ₹1,103,776,760

Delay rate by supplier tier:
reliability_tier
high       7.2% delayed
low       41.0% delayed
medium    20.7% delayed


In [9]:
# ============================================================
# TABLE 4: INVENTORY SNAPSHOTS
# Weekly snapshots | Jan 2023 - Dec 2024
# 3 Warehouses: Mumbai, Delhi, Bengaluru
# Some SKUs intentionally designed to have dead stock risk
# ============================================================

warehouses = ['WH-MUM', 'WH-DEL', 'WH-BLR']

# Warehouse demand weight — Mumbai sells most, BLR sells least
warehouse_demand_weight = {
    'WH-MUM': 1.0,
    'WH-DEL': 0.85,
    'WH-BLR': 0.65
}

# SKUs with intentionally slow movement (dead stock candidates)
# These are C-category items that barely sell
slow_moving_skus = ['SKU041', 'SKU042', 'SKU043', 'SKU044', 'SKU045',
                    'SKU046', 'SKU047', 'SKU048', 'SKU049', 'SKU050']

inventory_snapshots = []

# Weekly snapshots from Jan 2023 to Dec 2024
snapshot_date = datetime(2023, 1, 1)
end_date      = datetime(2024, 12, 31)

# Initial stock for each SKU per warehouse
initial_stock = {}
for sku in df_products['sku_id']:
    for wh in warehouses:
        if sku in slow_moving_skus:
            initial_stock[(sku, wh)] = random.randint(800, 2000)  # High initial, slow movement
        else:
            initial_stock[(sku, wh)] = random.randint(300, 1500)

current_stock = initial_stock.copy()

while snapshot_date <= end_date:
    for sku_row in df_products.itertuples():
        sku_id    = sku_row.sku_id
        category  = sku_row.category
        rop       = sku_row.reorder_point_units
        moq       = sku_row.minimum_order_qty

        for wh in warehouses:
            wh_weight = warehouse_demand_weight[wh]
            stock_key = (sku_id, wh)

            # Base weekly demand per category
            base_demand_map = {
                'Staples':       random.randint(80, 200),
                'Personal Care': random.randint(40, 120),
                'Snacks':        random.randint(60, 180),
                'Beverages':     random.randint(50, 150),
                'Household':     random.randint(30, 100)
            }

            # Seasonal multiplier — Diwali (Oct-Nov) spike
            month = snapshot_date.month
            if month in [10, 11]:
                seasonal_mult = random.uniform(1.4, 1.8)
            elif month in [12, 1]:
                seasonal_mult = random.uniform(1.1, 1.3)
            elif month in [4, 5]:
                seasonal_mult = random.uniform(0.8, 1.0)
            else:
                seasonal_mult = random.uniform(0.9, 1.1)

            # Slow moving SKUs sell very little
            if sku_id in slow_moving_skus:
                weekly_demand = int(random.randint(2, 15) * wh_weight)
            else:
                weekly_demand = int(
                    base_demand_map[category] * wh_weight * seasonal_mult
                )

            opening_stock = current_stock[stock_key]

            # Units received this week (replenishment trigger)
            units_received = 0
            if opening_stock <= rop:
                units_received = moq
                current_stock[stock_key] += units_received

            # Can only sell what's available
            units_sold    = min(weekly_demand, current_stock[stock_key])
            closing_stock = current_stock[stock_key] - units_sold
            closing_stock = max(closing_stock, 0)

            # Update running stock
            current_stock[stock_key] = closing_stock

            # Days of inventory remaining
            avg_daily_demand = weekly_demand / 7 if weekly_demand > 0 else 1
            days_of_inventory = round(closing_stock / avg_daily_demand, 1) if avg_daily_demand > 0 else 999

            # Stockout flag
            stockout_flag = 1 if closing_stock == 0 else 0

            inventory_snapshots.append({
                'snapshot_date':    snapshot_date.strftime('%Y-%m-%d'),
                'sku_id':           sku_id,
                'warehouse_id':     wh,
                'opening_stock':    opening_stock,
                'units_received':   units_received,
                'units_sold':       units_sold,
                'closing_stock':    closing_stock,
                'days_of_inventory': days_of_inventory,
                'stockout_flag':    stockout_flag,
                'reorder_triggered': 1 if units_received > 0 else 0
            })

    # Move to next week
    snapshot_date += timedelta(weeks=1)

df_inventory = pd.DataFrame(inventory_snapshots)

df_inventory.to_csv("D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Raw/inventory_snapshots.csv", index=False)

print("✅ Inventory Snapshots table created successfully!")
print(f"Total rows: {len(df_inventory):,}")
print(f"\nWarehouses: {df_inventory['warehouse_id'].unique()}")
print(f"\nDate range: {df_inventory['snapshot_date'].min()} to {df_inventory['snapshot_date'].max()}")
print(f"\nStockout events: {df_inventory['stockout_flag'].sum():,}")
print(f"\nSlow moving SKU avg closing stock:")
slow = df_inventory[df_inventory['sku_id'].isin(slow_moving_skus)]
print(f"  {slow['closing_stock'].mean():.0f} units (vs normal SKUs: {df_inventory[~df_inventory['sku_id'].isin(slow_moving_skus)]['closing_stock'].mean():.0f} units)")

✅ Inventory Snapshots table created successfully!
Total rows: 15,750

Warehouses: ['WH-MUM' 'WH-DEL' 'WH-BLR']

Date range: 2023-01-01 to 2024-12-29

Stockout events: 0

Slow moving SKU avg closing stock:
  1380 units (vs normal SKUs: 1625 units)


In [10]:
# ============================================================
# TABLE 5: SALES ORDERS
# 2500+ rows | Jan 2023 - Dec 2024
# 4 channels: Amazon, Flipkart, Own Website, Modern Trade
# Seasonal spikes built in for Diwali, New Year, Summer
# ============================================================

channels = ['Amazon', 'Flipkart', 'Own Website', 'Modern Trade']

# Channel mix — Amazon dominates for D2C brands in India
channel_weights = [0.40, 0.30, 0.20, 0.10]

# Top cities for orders
cities = [
    'Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai',
    'Pune', 'Ahmedabad', 'Kolkata', 'Jaipur', 'Lucknow',
    'Surat', 'Chandigarh', 'Bhopal', 'Kochi', 'Nagpur'
]

# City weights — metros order more
city_weights = [0.18, 0.17, 0.14, 0.10, 0.09,
                0.07, 0.06, 0.05, 0.04, 0.03,
                0.02, 0.02, 0.01, 0.01, 0.01]

sales_orders = []
order_counter = 1

start_date = datetime(2023, 1, 1)
end_date   = datetime(2024, 12, 31)

# Generate ~2600 orders spread across 2 years
total_orders = 2600

for _ in range(total_orders):
    # Random order date
    days_range = (end_date - start_date).days
    order_date = start_date + timedelta(days=random.randint(0, days_range))
    month      = order_date.month

    # Seasonal demand multiplier
    if month in [10, 11]:      # Diwali season
        seasonal_mult = random.uniform(1.5, 2.0)
    elif month in [12, 1]:     # New Year / Winter
        seasonal_mult = random.uniform(1.2, 1.5)
    elif month in [4, 5, 6]:   # Summer — slow for some, fast for beverages
        seasonal_mult = random.uniform(0.8, 1.1)
    else:
        seasonal_mult = random.uniform(0.9, 1.2)

    # Pick channel and city
    channel = random.choices(channels, weights=channel_weights, k=1)[0]
    city    = random.choices(cities,   weights=city_weights,   k=1)[0]

    # Pick a random SKU
    sku_row       = df_products.sample(1).iloc[0]
    sku_id        = sku_row['sku_id']
    selling_price = sku_row['selling_price_inr']
    category      = sku_row['category']

    # Units ordered based on channel
    channel_qty_map = {
        'Amazon':       (5,  80),
        'Flipkart':     (5,  60),
        'Own Website':  (1,  20),
        'Modern Trade': (50, 500)   # Bulk orders
    }
    units_ordered = int(
        random.randint(*channel_qty_map[channel]) * seasonal_mult
    )

    # Channel-wise fulfillment rate
    channel_fulfillment = {
        'Amazon':       random.uniform(0.92, 1.00),
        'Flipkart':     random.uniform(0.90, 1.00),
        'Own Website':  random.uniform(0.95, 1.00),
        'Modern Trade': random.uniform(0.85, 0.98)
    }
    fulfillment_rate  = channel_fulfillment[channel]
    units_fulfilled   = int(units_ordered * fulfillment_rate)

    # Revenue calculations
    gross_revenue     = round(units_fulfilled * selling_price, 2)

    # Channel commission (platform fees)
    commission_map = {
        'Amazon':       0.18,
        'Flipkart':     0.15,
        'Own Website':  0.02,
        'Modern Trade': 0.10
    }
    commission_pct    = commission_map[channel]
    commission_amount = round(gross_revenue * commission_pct, 2)
    net_revenue       = round(gross_revenue - commission_amount, 2)

    # Return rate by category
    return_rate_map = {
        'Staples':       0.02,
        'Personal Care': 0.06,
        'Snacks':        0.03,
        'Beverages':     0.04,
        'Household':     0.05
    }
    return_rate   = return_rate_map[category]
    units_returned = int(units_fulfilled * return_rate)

    # Fulfillment status
    if units_fulfilled == units_ordered:
        fulfillment_status = 'Fulfilled'
    elif units_fulfilled == 0:
        fulfillment_status = 'Stockout'
    else:
        fulfillment_status = 'Partial'

    sales_orders.append({
        'order_id':           f'ORD{str(order_counter).zfill(6)}',
        'order_date':         order_date.strftime('%Y-%m-%d'),
        'sku_id':             sku_id,
        'category':           category,
        'channel':            channel,
        'city':               city,
        'units_ordered':      units_ordered,
        'units_fulfilled':    units_fulfilled,
        'units_returned':     units_returned,
        'selling_price_inr':  selling_price,
        'gross_revenue_inr':  gross_revenue,
        'commission_pct':     commission_pct,
        'commission_inr':     commission_amount,
        'net_revenue_inr':    net_revenue,
        'fulfillment_rate':   round(fulfillment_rate, 4),
        'fulfillment_status': fulfillment_status,
    })
    order_counter += 1

df_sales = pd.DataFrame(sales_orders)

# Sort by order date
df_sales = df_sales.sort_values('order_date').reset_index(drop=True)

df_sales.to_csv("D:/Projects/End-to-end projects/8. Supply Chain Intelligence/Data/Raw/sales_orders.csv", index=False)

print("✅ Sales Orders table created successfully!")
print(f"Total orders: {len(df_sales):,}")
print(f"\nChannel breakdown:")
print(df_sales['channel'].value_counts())
print(f"\nFulfillment status:")
print(df_sales['fulfillment_status'].value_counts())
print(f"\nTotal Gross Revenue: ₹{df_sales['gross_revenue_inr'].sum():,.0f}")
print(f"Total Net Revenue:   ₹{df_sales['net_revenue_inr'].sum():,.0f}")
print(f"Total Returns Value: ₹{(df_sales['units_returned'] * df_sales['selling_price_inr']).sum():,.0f}")
print(f"\nTop 5 cities by orders:")
print(df_sales['city'].value_counts().head())

✅ Sales Orders table created successfully!
Total orders: 2,600

Channel breakdown:
channel
Amazon          1054
Flipkart         813
Own Website      485
Modern Trade     248
Name: count, dtype: int64

Fulfillment status:
fulfillment_status
Partial      2574
Stockout       22
Fulfilled       4
Name: count, dtype: int64

Total Gross Revenue: ₹56,953,903
Total Net Revenue:   ₹49,542,424
Total Returns Value: ₹1,961,406

Top 5 cities by orders:
city
Mumbai       459
Delhi        431
Bengaluru    366
Hyderabad    274
Chennai      242
Name: count, dtype: int64
